---
title: Beginner's Guide exercise solution
short_title: Exercise solution
subject: Beginner Guide
subtitle: A worked solution for the Beginner's Guide exercise.
description: A worked solution for the Beginner's Guide exercise.
authors:
  - name: Muhammad Taufik
    github: taufik-shf
keywords:
  - open-data-cube
  - odc
  - xarray
  - plotting
  - spectral-indices
  - beginner-guide
  - exercise
---

This notebook works through the Beginner's Guide exercise. [^1]

[^1]: Tutorial notebooks update automatically; edits to a tutorial notebook may be overwritten on the next update. Keep a working copy in a separate file to preserve changes.

## A. Getting started

In [ ]:
from datacube import Datacube
from odc.geo.geom import point

dc = Datacube(app="beginners_guide_exercise")

## B. Load data

A `0.05`-degree buffer around the point defines the area of interest. Its bounds supply the `x` and `y` ranges for the query.

In [ ]:
latitude = -8.81
longitude = 116.008667

bbox = point(longitude, latitude, crs="EPSG:4326").buffer(0.05).boundingbox
bbox.explore()

The query loads annual GeoMAD data from 2022 to 2025 at 30-metre resolution. Kuta, Lombok lies in UTM zone 50S, so the output grid uses `EPSG:32750`.

In [ ]:
query = {
    "product": "s2_geomad_annual",
    "x": (bbox.left, bbox.right),
    "y": (bbox.bottom, bbox.top),
    "time": ("2022", "2025"),
    "measurements": ["red", "green", "blue", "nir"],
    "output_crs": "EPSG:32750",
    "resolution": (-30, 30),
}

ds = dc.load(**query)

## C. Inspect the dataset

These cells show the Dataset dimensions, data variables, and coordinate reference system.

In [ ]:
ds.sizes

In [ ]:
ds.data_vars

In [ ]:
ds.attrs["crs"]

## D. Plot a true-colour image

Stack the red, green, and blue measurements along a `band` dimension, then plot the first time slice.

In [ ]:
rgb = ds[["red", "green", "blue"]].to_array(dim="band").isel(time=0)
rgb.plot.imshow(vmin=0, vmax=3000)

## E. Calculate spectral indices

Convert the source bands to floating-point values before division, then add the indices to the Dataset.

In [ ]:
red = ds.red.astype("float32")
green = ds.green.astype("float32")
nir = ds.nir.astype("float32")

ds["ndvi"] = (nir - red) / (nir + red)
ds["ndwi"] = (green - nir) / (green + nir)

ds[["ndvi", "ndwi"]]

## F. Plot the indices

Use the same scale for every index plot so the values remain comparable.

In [ ]:
ds.ndvi.isel(time=0).plot(cmap="RdYlGn", vmin=-1, vmax=1)

In [ ]:
ds.ndwi.isel(time=0).plot(cmap="RdBu", vmin=-1, vmax=1)

In [ ]:
ds.ndvi.plot(col="time", col_wrap=2, cmap="RdYlGn", vmin=-1, vmax=1)

## G. Interpret the results

Vegetation usually has higher NDVI than bare ground or water. Positive NDWI often marks open water. Read the index plots alongside the true-colour image, then compare the annual NDVI panels on their shared scale.

Annual GeoMAD combines observations from a year into one composite. Differences between panels can reflect land-cover change or the observations contributing to each composite.

## H. Optional challenge

Facet NDWI across the same years.

In [ ]:
ds.ndwi.plot(col="time", col_wrap=2, cmap="RdBu", vmin=-1, vmax=1)

## I. Next steps

Compare these cells with the exercise, then adapt the point or time range and inspect the resulting changes.